In [1]:
from IPython.display import display,Markdown
display(Markdown("# Document splitter and Vector Store"))

# Document splitter and Vector Store

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from langchain_core.documents import Document

In [4]:
import json
data={}
with open('../data/articles.json','r') as f:
    data=json.load(f)

In [5]:
documents=[]
for key,value in zip(data.keys(),data.values()):
    documents.append(Document(metadata={"Article":key},page_content=f"{key} \n: {value}"))

In [16]:
# vector_store=FAISS.from_documents(documents,embedding=embeddings)

In [17]:
# vector_store.save_local("../data/constitution.faiss")

In [6]:
vector_store=FAISS.load_local("../data/constitution.faiss",embeddings=embeddings,allow_dangerous_deserialization=True)

In [7]:
import json
data={}
with open('../data/penal_code_sections.json','r') as f:
    data=json.load(f)

In [8]:
documents=[]
for key,value in data.items():
    documents.append(Document(metadata={"Section":key},page_content=f"{key} \n{value}"))

In [9]:
print(documents[0].page_content)

IPC Section 6 - Definitions in the Code to be understood subject to exceptions 
Throughout this Code every definition of an Offence, every penal provision and every illustration of every such definition or penal provision, shall be understood subject to the exceptions contained in the Chapter entitled ?General Exceptions?, though those exceptions are not repeated in such definition, penal provision, or illustration.


In [11]:
# vector_store.add_documents(documents=documents)

In [12]:
# vector_store.save_local("../data/constitution_and_ipc.faiss")

In [13]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [14]:
res=retriever.invoke("murder related law",k=3)
for i in res:
    print(i.page_content)
    print("\n\n")

Article 21 of Indian Constitution 
: Protection of life and personal liberty.—No person shall be deprived of his life or personal liberty except according to procedure established by law.



Article 134 of Indian Constitution 
: Appellate jurisdiction of Supreme Court in regard to criminal matters.—
(1) An appeal shall lie to the Supreme Court from any judgment, final order or sentence in a criminal proceeding of a High Court in the territory of India if the High Court—
(a) has on appeal reversed an order of acquittal of an accused person and sentenced him to death; or
(b) has withdrawn for trial before itself any case from any court subordinate to its authority and has in such trial convicted the accused person and sentenced him to death; or
(c) [certifies under article 134A] that the case is a fit one for appeal to the Supreme Court:
Provided that an appeal under sub-clause (c) shall lie subject to such provisions as may be made in that behalf under clause (1) of article 145 and to s

In [15]:
from IPython.display import display,Markdown
display(Markdown("# Architecture"))

# Architecture

In [35]:
from typing import TypedDict,List,Literal
from pydantic import Field

class schema(TypedDict):
    retrieval_required:bool
    user_query:str
    k:int=Field(default=3)
    retrieved_contexts:List[str]
    answer_for_query:str
    relevent_contexts:List[str]
    generated_response:str
    is_grounded:bool
    is_supported:bool
    evidence:str
    retriever
    

In [21]:
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

class schema_for_retrieval_decider_node(BaseModel):
    retrieval_required:bool=Field(...,description="True if retrieval required false otherwise")

parser_for_retrieval_decider_node=PydanticOutputParser(pydantic_object=schema_for_retrieval_decider_node)

sys_prompt_for_retrieval_decider_node=f"You are an AI Assistant . In retriever Indian Penal Code and Consititution of India is present . User will give you some query your task is to determine whether retrieval is required for this query or not \n Output Format -{parser_for_retrieval_decider_node.get_format_instructions()}"


def retrieval_decider_node(state:schema):
    inp=[SystemMessage(content=sys_prompt_for_retrieval_decider_node),HumanMessage(content=f"User Query - {state['user_query']}")]
    res=parser_for_retrieval_decider_node.invoke(main_model.invoke(inp).content).retrieval_required
    return {'retrieval_required':res}

In [22]:
def retrieve(state:schema):
    retrieved_contexts=state['retriever'].invoke(state['user_query'],k=state['k'])
    return {'retrieved_contexts':[i.page_content for i in retrieved_contexts]}

In [23]:
def direct_generation(state:schema):
    res=main_model.invoke(state['user_query']).content
    return {"answer_for_query":res}

In [24]:
class schema_for_is_relevant_node(BaseModel):
    is_relevant_context:bool
parser_for_is_relevant_node=PydanticOutputParser(pydantic_object=schema_for_is_relevant_node)

def is_relevant_node(state:schema):
    contexts=state['retrieved_context']
    sys_prompt=SystemMessage(content=f"""User will feed you a query  and a retrieved context for that query your task is to tell whether retrieved context is relevent for answering the given query or not \n Output format - {parser_for_is_relevant_node.get_format_instructions()}""")
    hmn_prompt=f"Query - {state['user_query']}"
    lst=[]
    for context in contexts:
        hmn_prompt_dash=hmn_prompt+f"\n Context - {context}"
        res=parser_for_is_relevant_node.invoke(model.invoke([sys_prompt,HumanMessage(content=hmn_prompt_dash)]).content)

        lst.append(res.is_relevant_context)

    return {'relevent_contexts':contexts[i] for i in range(len(contexts)) if lst[i]}

In [30]:
class schema_for_answer_from_context_node(BaseModel):
    response:str=Field(...,description="Response for given query")
    
parser_for_answer_from_context_node=PydanticOutputParser(pydantic_object=schema_for_answer_from_context_node)

def answer_from_context_node(state:schema):
    contexts=state['relevent_contexts']
    sys_prompt_for_answer_from_context_node=SystemMessage(content=f"""User will give you a query and context for query your work is to give answer user query based on the context provided. \n Output format - {parser_for_answer_from_context_node.get_format_instructions()}""")
    
    context=""
    for i in contexts:
        context+=i
        context+="\n"
        
    hmn_prompt=f"Query - {state['user_query']} \n\n Contexts - {context}"
    inp=[SystemMessage(content=sys_prompt_for_answer_from_context_node),HumanMessage(content=hmn_prompt)]

    res=parser_for_answer_from_context_node.invoke(main_model.invoke(inp).content).response
    return {"generated_response":res}

In [34]:
class schema_for_check_answer_grounded_node(BaseModel):
    is_grounded:Literal['fully_supported','not_fully_supported']
    evidence:str=Field(...,description="Proof that answer is not supported by given contexts")

parser_for_schema_for_check_answer_grounded_node=PydanticOutputParser(pydantic_object=schema_for_check_answer_grounded_node)

def check_answer_grounded_node(state:schema):
    contexts=state['relevent_contexts']
    sys_prompt=SysteMessage(content=f"You are verifying whether the ANSWER is supported by the CONTEXT.\n Output format - {parser_for_support_check_node.get_format_instructions()}")
    context=""
    for i in contexts:
        context+=i
        context+="\n"

    human_pr=HumanMessage(content=f"Answer - {state['generated_response']} \n Contexts - {x}")

    res=parser_for_schema_for_check_answer_grounded_node.invoke(main_model.invoke([sys_prompt,human_pr]).content)
    
    return {'is_grounded':res.is_grounded,'evidence':res.evidence}

In [32]:
class schema_for_revise_answer_node(BaseModel):
    revised_response:str=Field(...,description="Response for given query")
    
parser_for_revise_answer_node=PydanticOutputParser(pydantic_object=schema_for_revise_answer_node)


def revise_answer_node(state:schema):
    contexts=state['relevent_contexts']
    context=""
    for i in contexts:
        context+=i
        context+="\n"
    generated_response=state['generated_response']
    user_query=state['user_query']
    evidence=state['evidence']
    
    sys_prompt=SystemMessage(content=f"User will give you a query ,an answer for the given query, context for the given query, response generated by llm and a evidence that the response is not fully supported by the given contexts Your task is to revise the answer such that revised answer is fully suported by the given contexts\n Output  format- {parser_for_revise_answer_node.get_format_instructions()}")
    
    
    human_pr=HumanMessage(content=f"Query - {user_query} \n\n Generated Response - {generated_response} \n\n Contexts - {x} \n\n Evidence - {evidence}")

    revised_answer=parser_for_revise_answer_node.invoke(critic_model.invoke([sys_prompt,human_pr]).content)
    
    return {'generated_response':res.revised_response,'max_retry':state['max_retry']+1}

In [33]:
class schema_for_is_answer_useful_node(BaseModel):
    is_useful:str=Field(...,description="Boolean response whether answer is useful or not")

parser_for_is_answer_useful_node=PydanticOutputParser(pydantic_object=schema_for_is_answer_useful_node)

def is_answer_useful(state:schema):
    user_query=state['user_query']
    generated_response=state['generated_response']

    sys_prompt=SystemMessage(content=f"User will give you a query and a response of the query generated by llm your task is to tell whether the response solves users query or not. Output format - {parser_for_is_answer_useful_node.get_format_instructions()}")
    human_pr=HumanMessage(content=f"Query - {user_query} \n\n Generated Response - {generated_response}")

    res=parser_for_is_answer_useful_node.invoke(judge_llm.invoke([sys_prompt,human_pr]).content)
    return {'is_supported':res.is_useful}

In [ ]:
class schema_for_rewrite_query_node(BaseModel):
    updated_query:str

def rewrite_query_node(state:schema):
    